# Neighbor Joining Implementation

## Imports

In [1]:
import subprocess
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
from pathlib import Path
from scipy.cluster.hierarchy import dendrogram

## Data preprocessing

In [ ]:
# Returns a dict where species_seq[species name] --> nucleotide sequence . species_seq[str] --> str
def extractFromData(data_path):
  data_dir = Path(data_path)
  species_seq = {}
  firstN = 6
  i = 0
  for species_dir in sorted(data_dir.iterdir()):
    if firstN > 0 and i == firstN:
      break
    i += 1
    gene_file = species_dir / 'ncbi_dataset' / 'data' / 'gene.fna'
    split_species = str(species_dir).split('/')
    species_array = split_species[len(split_species) - 1].split("_")
    if gene_file.exists():
      for record in SeqIO.parse(str(gene_file), 'fasta'):
        species_name = species_array[0] + " " + species_array[1]
        species_seq[species_name] = record.seq
  return species_seq

In [11]:
# Takes in an array of strings and returns the multiple sequence alignment in the form of a "MultipleSeqAlignment" object
def createMSA(species_seq):
  sequences = []
  for species, seq in species_seq.items():
    species = species.replace(" ", "_")
    sequences.append(SeqRecord(Seq(seq), id=species))
  # sequences = [SeqRecord(Seq(strings[i]), id=f"seq{i}") for i in range(len(strings))]

  input_file = "input_sequences.fasta"
  output_file = "aligned_sequences.fasta"

  SeqIO.write(sequences, input_file, "fasta")

  command = [
      "clustalo",
      "-i", input_file,
      "-o", output_file,
      "--outfmt=fa",
      "--auto",
      "-v"
  ]

  try:
    result = subprocess.run(command, capture_output=True, text=True, check=True)
  except subprocess.CalledProcessError as e:
      print(f"Error running Clustal Omega: {e.stderr}")

  alignment = SeqIO.parse(output_file, "fasta")

  msa = []
  for record in alignment:
      msa.append(record)
  msa = MultipleSeqAlignment(msa)

  os.remove(input_file)
  os.remove(output_file)

  return msa

## Distance matrix calculation

In [4]:
# Takes in a multiple sequence alignment object and returns a tuple of the names of the ids and their respectictive dist matrix
def computeDistMatK80(msa):
  names = [line.id for line in msa]
  msa_list = [[nuc for nuc in line.seq] for line in msa]
  num_sequences = len(msa_list)
  dist_matrix = [[0 for i in range(num_sequences)] for i in range(num_sequences)]
  for seq_1 in range(num_sequences - 1):
    for seq_2 in range(seq_1 + 1, num_sequences):
      p = 0
      q = 0
      total = 0
      for index in range(len(msa_list[seq_1])):
        pair = set()
        pair.add(msa_list[seq_1][index])
        pair.add(msa_list[seq_2][index])
        if '-' not in pair:
          total += 1
          if (msa_list[seq_1][index] != msa_list[seq_2][index]):
            if ('A' in pair and 'G' in pair) or ('T' in pair and 'C' in pair):
              p += 1
            else:
              q += 1
      if total == 0:
        dist = 0
      else:
        p /= total
        q /= total
        l1 = 1 - (2 * p) - q
        l2 = 1 - (2 * q)
        if l1 < 0 or l2 < 0:
          dist = float('inf')
        else:
          dist = (-(1 / 2) * math.log(l1)) - ((1 / 4) * math.log(l2))
      dist_matrix[seq_1][seq_2] = dist
      dist_matrix[seq_2][seq_1] = dist

  return (names, dist_matrix)

## Hierarchical clustering and tree assembly

In [5]:
def naiveClosestIndices(dist_matrix, indicesInPlay):
  # print(dist_matrix)
  minVal = np.inf
  clust1 = 0
  clust2 = 0
  listIndices  = list(indicesInPlay)
  for i in listIndices:
    for j in listIndices:
      if i == j:
        continue
      if minVal > dist_matrix[i][j]:
        clust1 = i
        clust2 = j
        minVal = dist_matrix[i][j]
  return (clust1, clust2, minVal)

def hierarchClust(dist_matrix):
  dist_matrix = np.array(dist_matrix, dtype=float)

  nextClustVal = len(dist_matrix)
  indicesInPlay = set([i for i in range(nextClustVal)])
  tree = {}
  # nextClustVal = numClust
  for i in range(nextClustVal):
    tree[i] = None

  while len(indicesInPlay) > 1:
    clust1, clust2, val = naiveClosestIndices(dist_matrix, indicesInPlay)
    tree[nextClustVal] = ((clust1, clust2), val)
    # print(clust1, clust2, val)
    # print(type(dist_matrix))
    dist_matrix = np.append(dist_matrix, [np.zeros_like(dist_matrix[0])], axis=0)
    dist_matrix = np.append(dist_matrix, np.zeros((dist_matrix.shape[0], 1)), axis=1)
    indicesInPlay.remove(clust1)
    indicesInPlay.remove(clust2)
    for idx in indicesInPlay:
      val = .5 * (dist_matrix[idx, clust1] + dist_matrix[idx, clust2] - dist_matrix[clust1, clust2])
      dist_matrix[nextClustVal, idx] = dist_matrix[idx, nextClustVal] = val
      # np.append(dist_matrix[idx], [val], axis=0)
    # print(dist_matrix)
    indicesInPlay.add(nextClustVal)
    nextClustVal += 1
  root = nextClustVal - 1
  return (tree, root)

In [6]:
def assembleTree(tree, root, names):
    linkageMatrix = []

    firstInternal = len(names)

    for i in range(firstInternal, root + 1):
        (clust1, clust2), dist, size = tree[i]
        print((clust1, clust2), size)
        # spacingBtwNodes = 2.0
        linkageMatrix.append([clust1, clust2, dist, size])

    plt.figure(figsize=(15, 8))
    dendrogram(
        linkageMatrix,
        labels=names,
        leaf_rotation=45,
        leaf_font_size=10,
        distance_sort='descending',
        show_leaf_counts=True
    )
    plt.title("Hierarchical Clustering Dendrogram")
    plt.xlabel("Species")
    plt.ylabel("Distance")
    plt.show()

## Run data on algorithm

In [7]:
def computePhylogenyFromData(data_path):
  species_seq = extractFromData(data_path)
  print("finished extracting...", species_seq)
  msa = createMSA(species_seq)
  print("created msa...", msa)
  names, dist_matrix = computeDistMatK80(msa)
  print("computed distance matrix...", names, dist_matrix)
  tree, root = hierarchClust(dist_matrix)
  print("computed phylogeny...", tree, root)
  assembleTree(tree, root, names)
  print("tree assembled...")